# Screenshots (capturas de pantalla)
---

* Para adjuntar screenshots automáticamente en Allure cuando una prueba falla, se utilizan los hooks o accesorios del framework de pruebas
* Esto evita escribir código de captura en cada prueba individual

## Configuración Automática con Pytest (conftest.py)

* Crear un archivo llamado conftest.py en la raíz del proyecto de pruebas. Pytest leerá este archivo automáticamente para capturar la pantalla justo en el momento en que detecte un fallo

In [ ]:
import pytest
import allure
from selenium import webdriver

@pytest.fixture(scope="function")
def driver():
    """Fixture que inicializa y cierra el navegador para cada prueba."""
    # Inicializa el navegador (puedes usar Chrome, Firefox, etc.)
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    yield driver  # Aquí es donde se ejecuta la prueba real
    
    # --- Código Post-Prueba: Se ejecuta al terminar el test ---
    driver.quit()

@pytest.hookimpl(tryfirst=True, hookwrapper=True)
def pytest_runtest_makereport(item, call):
    """Hook de Pytest que intercepta el resultado de la prueba."""
    outcome = yield
    rep = outcome.get_result()
    
    # Verifica si la prueba falló durante la fase de ejecución ('call')
    if rep.when == "call" and rep.failed:
        try:
            # Obtiene el driver desde la fixture de la prueba actual
            fixture_driver = item.funcargs.get("driver")
            if fixture_driver:
                # Toma la captura en formato binario (PNG)
                screenshot = fixture_driver.get_screenshot_as_png()
                
                # Adjunta la imagen directamente al reporte de Allure
                allure.attach(
                    screenshot,
                    name="Captura de Pantalla en Fallo",
                    attachment_type=allure.attachment_type.PNG
                )
        except Exception as e:
            print(f"No se pudo tomar la captura de pantalla: {e}")

## Ejemplo de Prueba que Falla (test_login.py)

* Al escribir la prueba, simplemente inyectar la fixture driver
* Si la aserción falla, el código del archivo conftest.py se activará automáticamente

In [ ]:
import pytest
import allure

def test_login_invalido_debe_fallar(driver):
    driver.get("https://ejemplo.com")
    
    with allure.step("Ingresar credenciales incorrectas"):
        # Código de Selenium para interactuar con la página...
        pass
        
    with allure.step("Verificar mensaje de error"):
        # Forzamos un fallo para probar la captura automática
        mensaje_esperado = "Usuario o contraseña incorrectos"
        mensaje_actual = "Error inesperado"
        
        assert mensaje_actual == mensaje_esperado, "El mensaje de error no coincide"

## ¿Cómo se ve en Allure Report?

* Cuando se ejecutan las pruebas con ```pytest --alluredir=allure-results```
* Y se abre el reporte con ```allure serve allure-results```
* La prueba aparecerá marcada en rojo (🔴 Failed)
* Al hacer clic en el nombre de la prueba, se desplegará su detalle
* En la sección inferior, se verá un apartado desplegable llamado Attachments (Adjuntos)
* Al expandir "Captura de Pantalla en Fallo", Allure mostrará la imagen PNG incrustada directamente en la interfaz web, permitiéndo ver exactamente qué estaba en la pantalla del navegador al momento del error